# YAMNet End-to-End Fine-tuning (Kaggle)

**Project**: Bird Vocalization Classification (BirdCLEF)
**Architecture**: YAMNet (partial unfreeze) -> Attention Pooling -> Deep Dense Head -> 1126 classes
**Training**: Two-stage (frozen warm-up + end-to-end fine-tune), Focal Loss, MixUp, SpecAugment, noise augmentation

## Architecture

```
16kHz waveform (80000 samples)
    |
YAMNet MobileNetV1 (bottom layers frozen, top layers trainable)
    |
Frame embeddings (T, 1024)
    |
SpecAugment (training only: freq mask + time mask)
    |
Attention Pooling (learnable) -> (1024,)
    |
Dense(512) -> BN -> ReLU -> Dropout(0.3)
    |
Residual Block: proj(256) + [Dense(256)->BN->ReLU->Drop(0.2)->Dense(256)->BN->ReLU]
    |
Dense(1126, softmax)
```

## Two-Stage Training

| Stage | Encoder | Head LR | Encoder LR | Epochs | Purpose |
|-------|---------|---------|------------|--------|---------|
| 1 | Frozen | 1e-3 | - | 10 | Warm up classification head |
| 2 | Unfrozen (top layers) | 5e-4 | 1e-5 | 20 | End-to-end fine-tune |

## Improvements over frozen baseline

| Item | Frozen baseline | This notebook |
|------|----------------|---------------|
| Encoder | Fully frozen | Top layers fine-tuned |
| Pooling | mean+max+std (fixed) | Attention pooling (learnable) |
| Head | 2-layer Dense | 3-layer + BN + Residual |
| Loss | Cross-entropy | Focal Loss + Label Smoothing |
| Augmentation | Offline only | Online noise + MixUp + SpecAugment |
| Training | model.fit | Custom loop, differential LR |

In [ ]:
get_ipython().system("pip install resampy -q")
import re, os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import librosa
from sklearn.metrics import (accuracy_score, top_k_accuracy_score,
                             f1_score, balanced_accuracy_score)

class Cfg:
    SR = 16000
    CLIP_SECONDS = 5.0
    CLIP_SAMPLES = int(SR * CLIP_SECONDS)  # 80000

    INPUT_BASE = "/kaggle/input"
    OUT_DIR = "/kaggle/working/yamnet_e2e"
    os.makedirs(OUT_DIR, exist_ok=True)

    TRAIN_CSV = "02_train_full_weighted.csv"
    TEST_CSV = "03_test_holdout.csv"
    CLASS_WEIGHTS_CSV = "class_weights.csv"
    N_FOLDS = 5

    YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"
    AUDIO_ROOT_CANDIDATES = ("train_audio", "train_short_audio")

    # --- Stage 1: frozen encoder, train head ---
    S1_EPOCHS = 10
    S1_LR = 1e-3
    S1_PATIENCE = 6

    # --- Stage 2: end-to-end fine-tune ---
    S2_EPOCHS = 20
    S2_ENC_LR = 1e-5
    S2_HEAD_LR = 5e-4
    S2_PATIENCE = 8

    BATCH_SIZE = 32
    SEED = 42

    # --- Loss ---
    FOCAL_GAMMA = 2.0
    LABEL_SMOOTH = 0.1

    # --- Online augmentation ---
    NOISE_AUG_PROB = 0.5
    NOISE_SNR_LO = 5.0
    NOISE_SNR_HI = 15.0
    MIXUP_ALPHA = 0.2
    MIXUP_PROB = 0.5

    # --- SpecAugment on frame embeddings ---
    SA_FREQ = 100
    SA_TIME = 2
    SA_NF = 2
    SA_NT = 2

    # --- Head ---
    ATTN_UNITS = 128
    DROP1 = 0.3
    DROP2 = 0.2

    # --- Eval ---
    FREQ_BINS = [0, 15, 50, 200, float("inf")]
    FREQ_NAMES = ["tail(<15)", "low(15-49)", "mid(50-199)", "head(>=200)"]

    @staticmethod
    def fold_csvs(fold):
        return (f"cv_fold{fold}_train.csv", f"cv_fold{fold}_val.csv")

cfg = Cfg()
print("[Cfg] OK")

## Data Utilities

Kaggle mount scanning, CSV loading, audio path resolution, label mapping.
(Adapted from yamnet_frozen_lightgbm_export.ipynb)

In [ ]:
def scan_kaggle_inputs(input_base=cfg.INPUT_BASE):
    target_csvs = {cfg.TRAIN_CSV, cfg.TEST_CSV, cfg.CLASS_WEIGHTS_CSV}
    for f in range(1, cfg.N_FOLDS + 1):
        target_csvs.update(cfg.fold_csvs(f))
    year2root, csv_paths = {}, {}
    if not Path(input_base).exists():
        return year2root, [], csv_paths
    unknown_roots = []
    for root, dirs, files in os.walk(input_base):
        base = os.path.basename(root)
        if base in cfg.AUDIO_ROOT_CANDIDATES:
            rp = Path(root)
            m = re.search(r"(20[0-9]{2})", root)
            if m:
                year2root.setdefault(int(m.group(1)), rp)
            elif rp not in unknown_roots:
                unknown_roots.append(rp)
            dirs[:] = []
            continue
        for f in files:
            if f in target_csvs and f not in csv_paths:
                csv_paths[f] = Path(root) / f
    for r in unknown_roots:
        year2root.setdefault(-1, r)
    return year2root, list(year2root.values()), csv_paths


def parse_source_years(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return []
    return sorted({int(y) for y in re.findall(r"20[0-9]{2}", str(s))}, reverse=True)


def resolve_audio_path(row, year2root, all_roots):
    fname = str(row["filename"]).replace(chr(92), "/")
    primary_label = str(row["primary_label"])
    basename = os.path.basename(fname)
    roots_to_try = []
    for y in parse_source_years(row.get("source_year")):
        r = year2root.get(y)
        if r is not None and r not in roots_to_try:
            roots_to_try.append(r)
    for r in all_roots:
        if r not in roots_to_try:
            roots_to_try.append(r)
    for root in roots_to_try:
        for cand in (os.path.join(root, fname),
                     os.path.join(root, primary_label, basename),
                     os.path.join(root, basename)):
            if cand and os.path.exists(cand):
                return cand
    return None


def build_label_map(csv_paths):
    label_frames = []
    for fname in [cfg.TRAIN_CSV, cfg.TEST_CSV, cfg.fold_csvs(1)[1]]:
        if fname in csv_paths:
            label_frames.append(
                pd.read_csv(csv_paths[fname], usecols=["primary_label"])["primary_label"])
    classes = sorted(pd.concat(label_frames).astype(str).unique().tolist())
    label2idx = {c: i for i, c in enumerate(classes)}
    idx2label = {i: c for c, i in label2idx.items()}
    out = Path(cfg.OUT_DIR) / "label_map.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump({"label2idx": label2idx,
                    "idx2label": {str(k): v for k, v in idx2label.items()},
                    "classes": classes}, f, ensure_ascii=False, indent=2)
    print(f"[Label] {len(classes)} classes -> {out}")
    return classes, label2idx, idx2label


def compute_class_train_counts(csv_paths):
    if cfg.TRAIN_CSV not in csv_paths:
        return {}
    df = pd.read_csv(csv_paths[cfg.TRAIN_CSV], usecols=["primary_label"])
    return df["primary_label"].value_counts().to_dict()

## Audio Pipeline

Streaming audio loading with on-the-fly noise augmentation for training.
- Clean loading: OGG -> 16kHz mono -> pad/clip 5s -> peak normalize
- Noisy loading: clean + Gaussian noise at random SNR (5-15 dB), 50% probability
- tf.data pipeline with parallel decoding

In [ ]:
def _load_clean(path_str):
    # Load audio -> 16kHz mono -> pad/clip 5s -> peak normalize
    target = cfg.CLIP_SAMPLES
    try:
        y, _ = librosa.load(path_str, sr=cfg.SR, mono=True, res_type="kaiser_fast")
    except Exception:
        return np.zeros(target, dtype=np.float32)
    if len(y) == 0:
        return np.zeros(target, dtype=np.float32)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        s = (len(y) - target) // 2
        y = y[s:s + target]
    peak = np.max(np.abs(y)) + 1e-9
    return (y / peak).astype(np.float32)


def _add_noise(clean, snr_db):
    signal_power = np.mean(clean ** 2) + 1e-12
    noise_power = signal_power / (10.0 ** (snr_db / 10.0))
    noise = np.random.randn(len(clean)).astype(np.float32) * np.sqrt(noise_power)
    noisy = clean + noise
    peak = np.max(np.abs(noisy)) + 1e-9
    return (noisy / peak).astype(np.float32)


def _load_wf_py(path_tensor):
    return _load_clean(path_tensor.numpy().decode("utf-8"))


def _load_wf_aug_py(path_tensor):
    # Training-time noise augmentation: 50% prob, SNR in [5, 15] dB
    p = path_tensor.numpy().decode("utf-8")
    clean = _load_clean(p)
    if np.random.random() < cfg.NOISE_AUG_PROB:
        snr = np.random.uniform(cfg.NOISE_SNR_LO, cfg.NOISE_SNR_HI)
        return _add_noise(clean, snr)
    return clean


def _load_wf_snr_py(path_tensor, snr_tensor):
    # Deterministic noise at given SNR (for noise evaluation)
    p = path_tensor.numpy().decode("utf-8")
    snr = float(snr_tensor.numpy())
    clean = _load_clean(p)
    if np.all(clean == 0):
        return clean
    return _add_noise(clean, snr)


def load_wf_clean_tf(path):
    wf = tf.py_function(_load_wf_py, [path], tf.float32)
    wf.set_shape([cfg.CLIP_SAMPLES])
    return wf


def load_wf_aug_tf(path):
    wf = tf.py_function(_load_wf_aug_py, [path], tf.float32)
    wf.set_shape([cfg.CLIP_SAMPLES])
    return wf


def load_wf_snr_tf(path, snr):
    wf = tf.py_function(_load_wf_snr_py, [path, snr], tf.float32)
    wf.set_shape([cfg.CLIP_SAMPLES])
    return wf


def resolve_paths(df, year2root, all_roots):
    # Resolve audio file paths, skip missing
    filepaths, keep_idx = [], []
    for i, (_, row) in enumerate(df.iterrows()):
        p = resolve_audio_path(row, year2root, all_roots)
        if p is not None:
            filepaths.append(p)
            keep_idx.append(i)
    df_keep = df.iloc[keep_idx].reset_index(drop=True)
    return filepaths, df_keep


def create_dataset(filepaths, labels, weights, batch_size=32,
                   shuffle=True, augment=False, snr=None):
    # Build tf.data pipeline
    n = len(filepaths)
    if snr is not None:
        # Fixed SNR mode (noise evaluation)
        snrs = [float(snr)] * n
        ds = tf.data.Dataset.from_tensor_slices((filepaths, labels, weights, snrs))
        def _proc(p, lab, w, s):
            wf = load_wf_snr_tf(p, s)
            return wf, lab, w
    else:
        ds = tf.data.Dataset.from_tensor_slices((filepaths, labels, weights))
        load_fn = load_wf_aug_tf if augment else load_wf_clean_tf
        def _proc(p, lab, w):
            wf = load_fn(p)
            return wf, lab, w

    if shuffle:
        ds = ds.shuffle(n, seed=cfg.SEED, reshuffle_each_iteration=True)
    ds = ds.map(_proc, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, n

## Model Architecture

- **YAMNet encoder**: MobileNetV1, AudioSet pre-trained. Stage 1 frozen, Stage 2 top layers unfrozen.
- **SpecAugment**: Frequency + time masking on frame embeddings (training only).
- **Attention Pooling**: Learnable weighted sum over time frames (replaces fixed mean+max+std).
- **Deep Head**: Dense(512)->BN->ReLU->Drop + Residual block (proj + 2-layer) -> softmax.
- **Focal Loss**: gamma=2.0 + label smoothing=0.1 for long-tail mitigation.

In [ ]:
class AttentionPooling(tf.keras.layers.Layer):
    # Learnable attention pooling: (B, T, D) -> (B, D)
    def __init__(self, units=128, **kw):
        super().__init__(**kw)
        self.units = units

    def build(self, input_shape):
        d = int(input_shape[-1])
        self.W = self.add_weight("W", (d, self.units), initializer="glorot_uniform")
        self.b = self.add_weight("b", (self.units,), initializer="zeros")
        self.u = self.add_weight("u", (self.units, 1), initializer="glorot_uniform")

    def call(self, x):
        score = tf.tanh(tf.tensordot(x, self.W, [[-1], [0]]) + self.b)
        weights = tf.nn.softmax(tf.tensordot(score, self.u, [[-1], [0]]), axis=1)
        return tf.reduce_sum(x * weights, axis=1)


class SpecAugmentEmb(tf.keras.layers.Layer):
    # SpecAugment on frame embeddings (training only)
    def __init__(self, freq_mask=100, time_mask=2, nf=2, nt=2, **kw):
        super().__init__(**kw)
        self.fm, self.tm, self.nf, self.nt = freq_mask, time_mask, nf, nt

    def call(self, x, training=None):
        if not training:
            return x
        B, T, D = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2]
        # Frequency masking
        for _ in range(self.nf):
            f = tf.minimum(tf.random.uniform([], 0, self.fm + 1, dtype=tf.int32), D - 1)
            f0 = tf.random.uniform([], 0, tf.maximum(D - f, 1), dtype=tf.int32)
            mask = tf.concat([tf.ones([B, T, f0]), tf.zeros([B, T, f]),
                              tf.ones([B, T, tf.maximum(D - f0 - f, 0)])], axis=2)
            x = x * mask
        # Time masking
        for _ in range(self.nt):
            t = tf.minimum(tf.random.uniform([], 0, self.tm + 1, dtype=tf.int32), T - 1)
            t0 = tf.random.uniform([], 0, tf.maximum(T - t, 1), dtype=tf.int32)
            mask = tf.concat([tf.ones([B, t0, D]), tf.zeros([B, t, D]),
                              tf.ones([B, tf.maximum(T - t0 - t, 0), D])], axis=1)
            x = x * mask
        return x


class FocalLoss(tf.keras.losses.Loss):
    # Focal Loss with label smoothing; handles hard (int) and soft (MixUp) labels
    def __init__(self, gamma=2.0, ls=0.1, nc=1126, **kw):
        super().__init__(reduction="none", **kw)
        self.gamma, self.ls, self.nc = gamma, ls, nc

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        if y_true.shape.rank == 1:
            y_true = tf.one_hot(tf.cast(y_true, tf.int32), self.nc)
        nc_f = tf.cast(self.nc, tf.float32)
        y_s = y_true * (1.0 - self.ls) + self.ls / nc_f
        ce = -tf.reduce_sum(y_s * tf.math.log(y_pred), axis=-1)
        pt = tf.reduce_sum(y_s * y_pred, axis=-1)
        return tf.pow(1.0 - pt, self.gamma) * ce


def _select_frame_emb(out):
    # Select 1024-dim frame embeddings from YAMNet output
    if isinstance(out, dict):
        for k in ("frame_embeddings", "embedding", "embeddings"):
            v = out.get(k)
            if v is not None and hasattr(v, "shape") and len(v.shape) >= 2 and int(v.shape[-1]) == 1024:
                return v
        for v in out.values():
            if hasattr(v, "shape") and len(v.shape) >= 2 and int(v.shape[-1]) == 1024:
                return v
    if isinstance(out, (list, tuple)):
        for v in out:
            if hasattr(v, "shape") and len(v.shape) >= 2 and int(v.shape[-1]) == 1024:
                return v
        return out[1] if len(out) > 1 else out[0]
    return out


def build_e2e_model(num_classes):
    # Build end-to-end YAMNet model
    yamnet = hub.KerasLayer(cfg.YAMNET_HANDLE, trainable=False, name="yamnet")

    inp = tf.keras.Input(shape=(cfg.CLIP_SAMPLES,), name="waveform")

    def _fwd(x):
        return _select_frame_emb(yamnet(x))

    frame = tf.keras.layers.Lambda(_fwd, name="yamnet_fwd")(inp)       # (B, T, 1024)
    frame = SpecAugmentEmb(cfg.SA_FREQ, cfg.SA_TIME, cfg.SA_NF, cfg.SA_NT,
                           name="spec_aug")(frame)
    pooled = AttentionPooling(cfg.ATTN_UNITS, name="attn_pool")(frame)  # (B, 1024)

    x = tf.keras.layers.Dense(512, name="fc1")(pooled)
    x = tf.keras.layers.BatchNormalization(name="bn1")(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Dropout(cfg.DROP1)(x)

    res = tf.keras.layers.Dense(256, name="res_proj")(x)
    x2 = tf.keras.layers.Dense(256, name="fc2")(x)
    x2 = tf.keras.layers.BatchNormalization(name="bn2")(x2)
    x2 = tf.keras.layers.ReLU()(x2)
    x2 = tf.keras.layers.Dropout(cfg.DROP2)(x2)
    x2 = tf.keras.layers.Dense(256, name="fc3")(x2)
    x2 = tf.keras.layers.BatchNormalization(name="bn3")(x2)
    x2 = tf.keras.layers.ReLU()(x2)
    x = tf.keras.layers.Add(name="residual")([res, x2])

    out = tf.keras.layers.Dense(num_classes, activation="softmax", name="output")(x)
    model = tf.keras.Model(inp, out, name="yamnet_e2e")

    n_params = model.count_params()
    print(f"[Model] params={n_params:,}, output={model.output_shape}")
    return model, yamnet

## Training Engine

- Custom training loop with `tf.GradientTape`
- **Stage 1**: YAMNet frozen, only head trains (single optimizer, lr=1e-3)
- **Stage 2**: YAMNet unfrozen, differential LR (encoder 1e-5, head 5e-4)
- **MixUp**: 50% probability per batch, alpha=0.2, at waveform level
- **EarlyStopping** on validation Top-1 accuracy

In [ ]:
def make_train_step(model, yamnet_layer, loss_fn, enc_opt, head_opt,
                    num_classes, stage):

    @tf.function
    def step(x, y, sw):
        # MixUp (50% prob)
        if tf.random.uniform([]) < cfg.MIXUP_PROB:
            lam = tf.random.uniform([], 1.0 - cfg.MIXUP_ALPHA, 1.0)
            idx = tf.random.shuffle(tf.range(tf.shape(x)[0]))
            x = lam * x + (1.0 - lam) * tf.gather(x, idx)
            y_oh = tf.one_hot(tf.cast(y, tf.int32), num_classes)
            y = lam * y_oh + (1.0 - lam) * tf.gather(y_oh, idx)
            sw = lam * sw + (1.0 - lam) * tf.gather(sw, idx)

        with tf.GradientTape() as tape:
            pred = model(x, training=True)
            loss = tf.reduce_mean(loss_fn(y, pred) * sw)

        if stage == 2 and yamnet_layer.trainable:
            enc_refs = {v.ref() for v in yamnet_layer.trainable_variables}
            all_v = model.trainable_variables
            grads = tape.gradient(loss, all_v)
            enc_gv = [(g, v) for g, v in zip(grads, all_v)
                      if v.ref() in enc_refs and g is not None]
            head_gv = [(g, v) for g, v in zip(grads, all_v)
                       if v.ref() not in enc_refs and g is not None]
            if enc_gv:
                enc_opt.apply_gradients(enc_gv)
            if head_gv:
                head_gv_filtered = [(g, v) for g, v in head_gv if g is not None]
                head_opt.apply_gradients(head_gv_filtered)
        else:
            tv = model.trainable_variables
            grads = tape.gradient(loss, tv)
            valid = [(g, v) for g, v in zip(grads, tv) if g is not None]
            head_opt.apply_gradients(valid)

        return loss
    return step


def evaluate_model(model, ds):
    preds, labels, probs = [], [], []
    for x, y, sw in ds:
        p = model(x, training=False)
        preds.append(tf.argmax(p, axis=1).numpy())
        labels.append(y.numpy())
        probs.append(p.numpy())
    yt = np.concatenate(labels)
    yp = np.concatenate(preds)
    ypr = np.concatenate(probs)
    top1 = accuracy_score(yt, yp)
    try:
        top5 = top_k_accuracy_score(yt, ypr, k=5,
                                    labels=list(range(ypr.shape[1])))
    except Exception:
        top5 = 0.0
    mf1 = f1_score(yt, yp, average="macro", zero_division=0)
    bacc = balanced_accuracy_score(yt, yp)
    return {"top1": top1, "top5": top5, "macro_f1": mf1, "balanced_acc": bacc}


def stratified_metrics(y_true, y_pred, class_train_counts, label2idx):
    # Per-frequency-group accuracy
    idx2count = {}
    for cls_name, cnt in class_train_counts.items():
        if cls_name in label2idx:
            idx2count[label2idx[cls_name]] = cnt
    result = {}
    for gi in range(len(cfg.FREQ_BINS) - 1):
        lo, hi = cfg.FREQ_BINS[gi], cfg.FREQ_BINS[gi + 1]
        gname = cfg.FREQ_NAMES[gi]
        mask = np.array([lo <= idx2count.get(int(t), 0) < hi for t in y_true])
        if mask.sum() > 0:
            result[f"acc_{gname}"] = float(accuracy_score(y_true[mask], y_pred[mask]))
        else:
            result[f"acc_{gname}"] = float("nan")
    return result


def train_one_fold(fold, df_train, df_val, year2root, all_roots,
                   label2idx, num_classes, class_train_counts):
    print()
    print("=" * 60)
    print(f"  FOLD {fold}/{cfg.N_FOLDS}")
    print("=" * 60)

    # Resolve paths
    fps_tr, df_tr = resolve_paths(df_train, year2root, all_roots)
    fps_va, df_va = resolve_paths(df_val, year2root, all_roots)
    y_tr = [label2idx[str(l)] for l in df_tr["primary_label"]]
    y_va = [label2idx[str(l)] for l in df_va["primary_label"]]
    if "loss_class_weight" in df_tr.columns:
        w_tr = df_tr["loss_class_weight"].astype(float).tolist()
    else:
        w_tr = [1.0] * len(df_tr)
    w_va = [1.0] * len(df_va)

    train_ds, n_tr = create_dataset(fps_tr, y_tr, w_tr,
                                     batch_size=cfg.BATCH_SIZE,
                                     shuffle=True, augment=True)
    val_ds, n_va = create_dataset(fps_va, y_va, w_va,
                                   batch_size=cfg.BATCH_SIZE,
                                   shuffle=False, augment=False)
    print(f"[Data] train={n_tr}, val={n_va}")

    model, yamnet_layer = build_e2e_model(num_classes)
    loss_fn = FocalLoss(cfg.FOCAL_GAMMA, cfg.LABEL_SMOOTH, num_classes)
    fold_dir = Path(cfg.OUT_DIR) / f"fold{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # ===== Stage 1: Frozen encoder =====
    print(f"[S1] Frozen encoder, lr={cfg.S1_LR}, epochs={cfg.S1_EPOCHS}")
    yamnet_layer.trainable = False
    head_opt = tf.keras.optimizers.Adam(cfg.S1_LR)
    step_s1 = make_train_step(model, yamnet_layer, loss_fn,
                              None, head_opt, num_classes, stage=1)
    best, wait = 0.0, 0
    for ep in range(cfg.S1_EPOCHS):
        t0 = time.time()
        eloss, nb = 0.0, 0
        for x, y, sw in train_ds:
            eloss += step_s1(x, y, sw).numpy()
            nb += 1
        vm = evaluate_model(model, val_ds)
        dt = time.time() - t0
        print(f"  S1 ep{ep+1:02d} loss={eloss/max(nb,1):.4f} "
              f"v_top1={vm['top1']:.4f} v_top5={vm['top5']:.4f} ({dt:.0f}s)")
        if vm["top1"] > best:
            best, wait = vm["top1"], 0
            model.save_weights(str(fold_dir / "s1_best.weights.h5"))
        else:
            wait += 1
            if wait >= cfg.S1_PATIENCE:
                print(f"  S1 early stop ep{ep+1}")
                break
    model.load_weights(str(fold_dir / "s1_best.weights.h5"))
    print(f"[S1] done, best v_top1={best:.4f}")

    # ===== Stage 2: End-to-end fine-tune =====
    print(f"[S2] Unfreeze, enc_lr={cfg.S2_ENC_LR}, head_lr={cfg.S2_HEAD_LR}, "
          f"epochs={cfg.S2_EPOCHS}")
    yamnet_layer.trainable = True
    enc_opt = tf.keras.optimizers.Adam(cfg.S2_ENC_LR)
    head_opt2 = tf.keras.optimizers.Adam(cfg.S2_HEAD_LR)
    step_s2 = make_train_step(model, yamnet_layer, loss_fn,
                              enc_opt, head_opt2, num_classes, stage=2)
    best, wait = 0.0, 0
    for ep in range(cfg.S2_EPOCHS):
        t0 = time.time()
        eloss, nb = 0.0, 0
        for x, y, sw in train_ds:
            eloss += step_s2(x, y, sw).numpy()
            nb += 1
        vm = evaluate_model(model, val_ds)
        dt = time.time() - t0
        print(f"  S2 ep{ep+1:02d} loss={eloss/max(nb,1):.4f} "
              f"v_top1={vm['top1']:.4f} v_top5={vm['top5']:.4f} ({dt:.0f}s)")
        if vm["top1"] > best:
            best, wait = vm["top1"], 0
            model.save_weights(str(fold_dir / "e2e_best.weights.h5"))
        else:
            wait += 1
            if wait >= cfg.S2_PATIENCE:
                print(f"  S2 early stop ep{ep+1}")
                break
    model.load_weights(str(fold_dir / "e2e_best.weights.h5"))
    print(f"[S2] done, best v_top1={best:.4f}")

    model.save(str(fold_dir / "e2e_model.keras"))
    return model

## 5-Fold CV, Noise Robustness Evaluation, Inference Speed

- 5-fold CV with identical splits as the frozen baseline
- Noise evaluation: clean / 5dB / 0dB / -5dB Gaussian white noise
- Inference latency: e2e vs full pipeline (from disk)

In [ ]:
def full_evaluate(model, ds, num_classes, class_train_counts, label2idx):
    # Full evaluation: top1/5, macro_f1, balanced_acc, stratified
    preds, labels, probs = [], [], []
    for x, y, sw in ds:
        p = model(x, training=False)
        preds.append(tf.argmax(p, axis=1).numpy())
        labels.append(y.numpy())
        probs.append(p.numpy())
    yt = np.concatenate(labels)
    yp = np.concatenate(preds)
    ypr = np.concatenate(probs)
    m = {
        "top1_acc": float(accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
        "balanced_acc": float(balanced_accuracy_score(yt, yp)),
    }
    try:
        m["top5_acc"] = float(top_k_accuracy_score(
            yt, ypr, k=5, labels=list(range(num_classes))))
    except Exception:
        m["top5_acc"] = 0.0
    m.update(stratified_metrics(yt, yp, class_train_counts, label2idx))
    return m, yt, yp


def main_cv(csv_paths, year2root, all_roots, classes, label2idx, idx2label,
            class_train_counts):
    nc = len(classes)
    all_results = []
    models = {}

    for fold in range(1, cfg.N_FOLDS + 1):
        tr_csv, va_csv = cfg.fold_csvs(fold)
        df_train = pd.read_csv(csv_paths[tr_csv])
        df_val = pd.read_csv(csv_paths[va_csv])
        model = train_one_fold(fold, df_train, df_val, year2root, all_roots,
                               label2idx, nc, class_train_counts)
        models[fold] = model

        # Test evaluation (clean)
        df_test = pd.read_csv(csv_paths[cfg.TEST_CSV])
        fps_te, df_te = resolve_paths(df_test, year2root, all_roots)
        y_te = [label2idx[str(l)] for l in df_te["primary_label"]]
        w_te = [1.0] * len(df_te)
        test_ds, _ = create_dataset(fps_te, y_te, w_te,
                                     batch_size=cfg.BATCH_SIZE,
                                     shuffle=False, augment=False)
        m, _, _ = full_evaluate(model, test_ds, nc, class_train_counts, label2idx)
        m["fold"] = fold
        all_results.append(m)
        print(f"[Test] fold{fold}: top1={m['top1_acc']:.4f} top5={m['top5_acc']:.4f} "
              f"f1={m['macro_f1']:.4f}")

    df_res = pd.DataFrame(all_results)
    df_res.to_csv(os.path.join(cfg.OUT_DIR, "cv_per_fold.csv"), index=False)
    summary = []
    for col in df_res.columns:
        if col == "fold":
            continue
        vals = df_res[col].dropna()
        if len(vals) > 0:
            summary.append({"metric": col, "mean": float(vals.mean()),
                            "std": float(vals.std(ddof=0))})
    pd.DataFrame(summary).to_csv(os.path.join(cfg.OUT_DIR, "cv_summary.csv"), index=False)
    print("[CV] Summary:")
    for r in summary:
        print(f"  {r['metric']}: {r['mean']:.4f} +/- {r['std']:.4f}")
    return models


def noise_eval(models, csv_paths, year2root, all_roots, label2idx, num_classes,
               class_train_counts):
    print()
    print("=" * 60)
    print("  Noise Robustness Evaluation")
    print("=" * 60)
    df_test = pd.read_csv(csv_paths[cfg.TEST_CSV])
    fps_te, df_te = resolve_paths(df_test, year2root, all_roots)
    y_te = [label2idx[str(l)] for l in df_te["primary_label"]]
    w_te = [1.0] * len(df_te)

    snr_tiers = [("clean", None), ("5dB", 5.0), ("0dB", 0.0), ("-5dB", -5.0)]
    rows = []
    for fold in range(1, cfg.N_FOLDS + 1):
        if fold not in models:
            continue
        model = models[fold]
        row = {"fold": fold}
        for tier_name, snr_val in snr_tiers:
            ds, _ = create_dataset(fps_te, y_te, w_te,
                                    batch_size=cfg.BATCH_SIZE,
                                    shuffle=False, augment=False, snr=snr_val)
            m, _, _ = full_evaluate(model, ds, num_classes,
                                    class_train_counts, label2idx)
            for k, v in m.items():
                row[f"{tier_name}_{k}"] = v
            print(f"  fold{fold} {tier_name}: top1={m['top1_acc']:.4f}")
        rows.append(row)

    df_noise = pd.DataFrame(rows)
    df_noise.to_csv(os.path.join(cfg.OUT_DIR, "cv_noise_per_fold.csv"), index=False)

    # Append to summary
    sp = Path(cfg.OUT_DIR) / "cv_summary.csv"
    existing = pd.read_csv(sp) if sp.exists() else pd.DataFrame(columns=["metric","mean","std"])
    new_rows = []
    for col in df_noise.columns:
        if col == "fold":
            continue
        vals = df_noise[col].dropna()
        if len(vals) > 0:
            new_rows.append({"metric": col, "mean": float(vals.mean()),
                             "std": float(vals.std(ddof=0))})
    df_new = pd.DataFrame(new_rows)
    existing = existing[~existing["metric"].isin(df_new["metric"])]
    pd.concat([existing, df_new], ignore_index=True).to_csv(sp, index=False)
    print("[Noise] done")


def inference_test(models, csv_paths, year2root, all_roots, n_samples=50):
    print()
    print("=" * 60)
    print("  Inference Speed Test")
    print("=" * 60)
    df_test = pd.read_csv(csv_paths[cfg.TEST_CSV])
    fps_te, df_te = resolve_paths(df_test, year2root, all_roots)
    fps = fps_te[:n_samples]
    n = len(fps)
    print(f"[Inference] {n} samples")

    # Pre-load waveforms
    wfs = np.stack([_load_clean(p) for p in fps]).astype(np.float32)

    e2e_lat = []
    full_lat = []
    for fold in [1]:
        if fold not in models:
            continue
        model = models[fold]
        # Warm up
        for i in range(min(5, n)):
            model(wfs[i:i+1], training=False)
        # E2E latency (pre-loaded waveforms, includes YAMNet forward)
        t0 = time.perf_counter()
        for i in range(n):
            model(wfs[i:i+1], training=False)
        ms = (time.perf_counter() - t0) / n * 1000
        e2e_lat.append(ms)
        print(f"  fold{fold} e2e latency: {ms:.1f} ms/sample")
        # Full pipeline (from disk)
        for i in range(min(3, n)):
            wf = _load_clean(fps[i])
            model(wf[None, :], training=False)
        t0 = time.perf_counter()
        for i in range(n):
            wf = _load_clean(fps[i])
            model(wf[None, :], training=False)
        ms2 = (time.perf_counter() - t0) / n * 1000
        full_lat.append(ms2)
        print(f"  fold{fold} full pipeline: {ms2:.1f} ms/sample")

    pd.DataFrame([{
        "n_samples": n,
        "e2e_latency_ms": float(np.mean(e2e_lat)) if e2e_lat else 0,
        "full_latency_ms": float(np.mean(full_lat)) if full_lat else 0,
    }]).to_csv(os.path.join(cfg.OUT_DIR, "inference_metrics.csv"), index=False)
    print("[Inference] done")

## Main Entry Point

Sequential execution:
1. Scan Kaggle inputs + build label map
2. 5-fold CV (two-stage training per fold)
3. Noise robustness evaluation
4. Inference speed test

In [ ]:
def main():
    t0 = time.time()

    print("#" * 60)
    print("# Phase 0: Scan + Label Map")
    print("#" * 60)
    year2root, all_roots, csv_paths = scan_kaggle_inputs(cfg.INPUT_BASE)
    print(f"[Audio] {len(year2root)} year roots:")
    for y, r in sorted(year2root.items(), key=lambda kv: str(kv[0])):
        print(f"  {y} -> {r}")
    if not year2root:
        raise SystemExit("[Error] No train_audio directory found.")
    print(f"[CSV] {len(csv_paths)} files:")
    for f, p in sorted(csv_paths.items()):
        print(f"  {f} -> {p}")

    classes, label2idx, idx2label = build_label_map(csv_paths)
    class_train_counts = compute_class_train_counts(csv_paths)
    nc = len(classes)
    print(f"[Stats] {nc} classes, {len(class_train_counts)} in training")

    # Phase 1: 5-fold CV
    print()
    print("#" * 60)
    print("# Phase 1: 5-Fold CV (two-stage E2E training)")
    print("#" * 60)
    models = main_cv(csv_paths, year2root, all_roots, classes,
                     label2idx, idx2label, class_train_counts)

    # Phase 2: Noise evaluation
    noise_eval(models, csv_paths, year2root, all_roots, label2idx, nc,
               class_train_counts)

    # Phase 3: Inference
    inference_test(models, csv_paths, year2root, all_roots)

    elapsed = (time.time() - t0) / 60
    print()
    print("=" * 60)
    print(f"[Done] Total: {elapsed:.1f} min")
    print(f"[Output] {cfg.OUT_DIR}")
    print("  cv_per_fold.csv       - per-fold metrics")
    print("  cv_summary.csv        - mean +/- std")
    print("  cv_noise_per_fold.csv - noise metrics")
    print("  inference_metrics.csv - latency")
    print("  fold{N}/e2e_model.keras - saved models")


main()